# Lekcija 11 - Protokol modelnega konteksta (MCP)

**Protokol modelnega konteksta (MCP)** je odprt standard, ki agentom omogoča dinamično odkrivanje in uporabo orodij, virov in podatkovnih virov med izvajanjem. Namesto da bi orodja vdelali v agenta, MCP omogoča agentom povezavo z zunanjimi strežniki, ki na zahtevo razkrivajo zmogljivosti.

V tej lekciji se boste naučili:
- Kaj je MCP in zakaj je pomemben za agentske sisteme
- Kako deluje MCP arhitektura odjemalec-strežnik
- Kako zgraditi agente, ki uporabljajo MCP-slog odkrivanja orodij


## Namestitev

**Predpogoji:**
- Projekt Microsoft Foundry z nameščenim modelom
- Za prijavo zaženite `az login`


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kaj je protokol za kontekst modela (MCP)?

MCP določa standarden način, kako lahko AI agenti odkrijejo in komunicirajo z zunanjimi orodji in viri podatkov:

- **MCP strežnik**: Izpostavlja orodja, vire in pozive prek standardnega protokola
- **MCP odjemalec**: Okolje izvajanja agentov, ki se povezuje s strežniki in odkriva razpoložljive zmogljivosti
- **Dinamično odkrivanje**: Agenti ne potrebujejo vnaprej kodiranih orodij — odkrijejo, kaj je na voljo med izvajanjem

To je močno za gradnjo razširljivih sistemov agentov, kjer je mogoče nove zmogljivosti dodajati brez sprememb kode agenta.


## Kako deluje MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP odjemalec) se poveže z MCP strežnikom
2. Strežnik odgovori s seznamom razpoložljivih orodij in njihovih shem
3. Agent lahko nato pokliče katerokoli odkrito orodje med svojim sklepanjem
4. Rezultati se vrnejo po istem protokolu


## Simulacija odkrivanja MCP orodja

Ker pravi MCP strežnik zahteva tečeči strežniški proces, bomo vzorec prikazali z uporabo funkcij `@tool`, ki simulirajo storitev namestitev, povezano z MCP.

V produkciji bi bila ta orodja dinamično odkrita z MCP strežnika, ne pa lokalno definirana.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Izdelava agenta z orodji v stilu MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP v proizvodnji

V proizvodnem okolju MCP omogoča zmogljive vzorce:

- **Dinamično odkrivanje orodij**: Agentii se povežejo s strežniki MCP in odkrijejo orodja med izvajanjem
- **Ločena arhitektura**: Ponudniki orodij se lahko posodabljajo neodvisno od agenta
- **Deljenje med organizacijami**: Skupine lahko preko strežnikov MCP omogočijo zmogljivosti, ki jih lahko uporablja vsak agent
- **Podpora za Microsoft Agent Framework**: MAF ima vgrajeno podporo za MCP klienta preko integracije `mcp`

Za uporabo pravega MCP strežnika z MAF se povežete prek `hosted_mcp_tool()` ali integracije MCP klienta.

**Več informacij:**
- [Specifikacija MCP](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework podpora za MCP](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Povzetek

V tej lekciji ste se naučili:
- **MCP** je odprt standard za dinamično odkrivanje orodij med agenti in ponudniki orodij
- **odjemalsko-strežniška arhitektura** omogoča agentom odkrivanje zmogljivosti v času izvajanja
- MCP omogoča **razširljive, ločene agentske sisteme**, kjer je mogoče orodja dodajati brez sprememb kode
- Microsoft Agent Framework zagotavlja **vgrajeno podporo MCP** za proizvodno uporabo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
